# PCA-8 + 100× decimation + flattop → single H5

**Pipeline:** ecei_mc decimated H5 (already 10× from 1 MHz → 100 kHz) → crop to flattop → additional 10× decimation (total 100× → 10 kHz) → streaming Welford PCA (160 ch → 8 PC) → tile into subsequences → save **all** splits into one H5.

**Output:** `/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/pca8_100x_flattop/all_data.h5`

**Subsequence shape:** `(8, 7812)` — small enough to load the entire dataset into memory.

**Flattop logic** (from `disruptcnn/dataset_original.py`):
- `t_flat_stop = t_flat_start + t_flat_last` (shot list convention)
- Disruptive: start = t_flat_start, end = max(tdisrupt, min(tlast, t_flat_stop))
- Clear: start = t_flat_start, end = min(tlast, t_flat_stop)
- Skip shots with NaN t_flat_start or t_flat_stop ≤ t_flat_start

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────
BASE = Path("/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc")
CLEAR_DECIMATED = BASE / "clear_decimated"
DISRUPT_DECIMATED = BASE / "disrupt_decimated"

# Shot lists (for flattop + t_disrupt info)
DISRUPT_SHOT_LIST = Path("disruptcnn/shots/d3d_disrupt_ecei.final.txt")
CLEAR_SHOT_LIST = Path("disruptcnn/shots/d3d_clear_ecei.final.txt")

# Output
OUT_DIR = BASE / "pca8_100x_flattop"
OUT_H5 = OUT_DIR / "all_data.h5"

# Decimation: ecei_mc is already 10× decimated; we do another 10× → total 100×
DATA_STEP = 10          # existing decimation (raw 1 MHz → 100 kHz in H5)
EXTRA_DECIMATE = 10     # additional decimation (100 kHz → 10 kHz)
TOTAL_DECIMATE = DATA_STEP * EXTRA_DECIMATE  # 100

# Subsequence params (in raw 1 MHz sample space, then divided by TOTAL_DECIMATE)
NSUB_RAW = 781_250
STRIDE_RAW = 481_090
T_SUB = NSUB_RAW // TOTAL_DECIMATE   # 7812
STRIDE = STRIDE_RAW // TOTAL_DECIMATE  # 4810

# PCA
N_PCA = 8
CHANNELS = 20 * 8  # 160

# Labels
TWARN_MS = 300.0
TWARN_RAW = int(TWARN_MS * 1000)  # 300,000 raw samples
TWARN_DEC = TWARN_RAW // DATA_STEP  # 30,000 in decimated space

# Splits
TRAIN_FRAC, VAL_FRAC = 0.8, 0.1
RANDOM_SEED = 42

# Shot list columns
COL_SHOT, COL_TSTART, COL_TLAST, COL_DT = 0, 2, 3, 4
COL_T_FLAT_START, COL_T_FLAT_LAST, COL_TDISRUPT = 6, 7, 8

print(f"T_SUB = {T_SUB}, STRIDE = {STRIDE}")
print(f"Output: {OUT_H5}")

## 1. Parse shot lists → flattop boundaries + t_disrupt

Read both disrupt and clear shot list files. Compute flattop start/end in **decimated** sample space (100 kHz). Skip shots with invalid flattop (NaN or t_flat_stop ≤ t_flat_start).

In [ ]:
def parse_shot_list(path: Path) -> pd.DataFrame:
    """Parse DisruptCNN-format shot list → DataFrame with flattop info in decimated samples."""
    data = np.loadtxt(path, skiprows=1)
    if data.ndim == 1:
        data = data[np.newaxis, :]
    df = pd.DataFrame({
        "shot": data[:, COL_SHOT].astype(int),
        "tstart_ms": data[:, COL_TSTART],
        "tlast_ms": data[:, COL_TLAST],
        "dt_ms": data[:, COL_DT],
        "t_flat_start_ms": data[:, COL_T_FLAT_START],
        "t_flat_last_ms": data[:, COL_T_FLAT_LAST],
        "tdisrupt_ms": data[:, COL_TDISRUPT],
    })
    # t_flat_stop = t_flat_start + t_flat_last (shot list convention)
    df["t_flat_stop_ms"] = df["t_flat_start_ms"] + df["t_flat_last_ms"]

    # Convert to decimated sample indices (100 kHz): idx = (t_ms - tstart_ms) / dt_ms / DATA_STEP
    df["flat_start_dec"] = ((df["t_flat_start_ms"] - df["tstart_ms"]) / df["dt_ms"] / DATA_STEP).astype(int)
    df["flat_stop_dec"] = ((df["t_flat_stop_ms"] - df["tstart_ms"]) / df["dt_ms"] / DATA_STEP).astype(int)
    df["tlast_dec"] = ((df["tlast_ms"] - df["tstart_ms"]) / df["dt_ms"] / DATA_STEP).astype(int)
    df["tdisrupt_dec"] = np.where(
        df["tdisrupt_ms"] > 0,
        ((df["tdisrupt_ms"] - df["tstart_ms"]) / df["dt_ms"] / DATA_STEP).astype(int),
        -1,
    )
    return df


df_disrupt = parse_shot_list(DISRUPT_SHOT_LIST)
df_clear = parse_shot_list(CLEAR_SHOT_LIST)
print(f"Disrupt shot list: {len(df_disrupt)} shots")
print(f"Clear shot list:   {len(df_clear)} shots")

# Index by shot for fast lookup
disrupt_info = df_disrupt.set_index("shot")
clear_info = df_clear.set_index("shot")

## 2. List H5 files, match with shot lists, stratified 80/10/10 split

In [ ]:
def list_h5_shots(root: Path) -> list[int]:
    if not root.exists():
        return []
    return sorted(int(p.stem) for p in root.glob("*.h5") if p.stem.isdigit())


def stratified_split(shot_ids: list[int], train_frac: float, val_frac: float, seed: int):
    n = len(shot_ids)
    if n == 0:
        return [], [], []
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    return (
        [shot_ids[i] for i in perm[:n_train]],
        [shot_ids[i] for i in perm[n_train:n_train + n_val]],
        [shot_ids[i] for i in perm[n_train + n_val:]],
    )


# List H5 files on disk, keep only those with valid flattop in shot list
disrupt_on_disk = list_h5_shots(DISRUPT_DECIMATED)
clear_on_disk = list_h5_shots(CLEAR_DECIMATED)

def filter_valid_flattop(shots_on_disk, shot_info_df):
    """Keep shots that exist on disk AND have valid flattop in shot list."""
    valid = []
    for s in shots_on_disk:
        if s not in shot_info_df.index:
            continue
        row = shot_info_df.loc[s]
        if np.isnan(row["t_flat_start_ms"]) or row["flat_stop_dec"] <= row["flat_start_dec"]:
            continue
        valid.append(s)
    return valid

disrupt_valid = filter_valid_flattop(disrupt_on_disk, disrupt_info)
clear_valid = filter_valid_flattop(clear_on_disk, clear_info)
print(f"Disrupt: {len(disrupt_on_disk)} on disk → {len(disrupt_valid)} with valid flattop")
print(f"Clear:   {len(clear_on_disk)} on disk → {len(clear_valid)} with valid flattop")

# Stratified split
disrupt_train, disrupt_val, disrupt_test = stratified_split(disrupt_valid, TRAIN_FRAC, VAL_FRAC, RANDOM_SEED)
clear_train, clear_val, clear_test = stratified_split(clear_valid, TRAIN_FRAC, VAL_FRAC, RANDOM_SEED)
print(f"Disrupt → train {len(disrupt_train)}, val {len(disrupt_val)}, test {len(disrupt_test)}")
print(f"Clear   → train {len(clear_train)}, val {len(clear_val)}, test {len(clear_test)}")

## 3. Helper: load a shot's flattop region, decimate by 10×

Load `(20, 8, T_dec)` from H5 → crop to flattop → decimate `[::10]` → reshape to `(T_100x, 160)`. This is the common loading step used by both PCA fitting and the final save.

In [ ]:
def get_flattop_bounds(shot: int, is_clear: bool) -> tuple[int, int, int]:
    """Return (flat_start_dec, flat_end_dec, tdisrupt_dec) for a shot.

    Flattop logic (matches disruptcnn/dataset_original.py):
    - Disruptive: end = max(tdisrupt, min(tlast, t_flat_stop))
    - Clear: end = min(tlast, t_flat_stop)
    tdisrupt_dec is in decimated space; -1 for clear shots.
    """
    info = disrupt_info if not is_clear else clear_info
    row = info.loc[shot]
    flat_start = int(row["flat_start_dec"])
    flat_stop = int(row["flat_stop_dec"])
    tlast = int(row["tlast_dec"])
    tdis = int(row["tdisrupt_dec"])

    if is_clear or tdis < 0:
        flat_end = min(tlast, flat_stop)
        tdis_local = -1
    else:
        flat_end = max(tdis, min(tlast, flat_stop))
        tdis_local = tdis  # absolute in decimated space
    return flat_start, flat_end, tdis_local


def load_flattop_100x(root: Path, shot: int, is_clear: bool) -> tuple[np.ndarray, int]:
    """Load one shot, crop to flattop, decimate 10×.

    Returns:
        X: (T_100x, 160) float64 — flattened channels, extra-decimated
        tdisrupt_100x: disruption index in 100× space relative to flattop start; -1 if clear
    """
    flat_start, flat_end, tdis_dec = get_flattop_bounds(shot, is_clear)
    with h5py.File(root / f"{shot}.h5", "r") as f:
        T_file = f["LFS"].shape[-1]
        # Clamp to file length
        flat_start = max(0, min(flat_start, T_file))
        flat_end = max(flat_start, min(flat_end, T_file))
        data = np.asarray(f["LFS"][..., flat_start:flat_end], dtype=np.float64)  # (20, 8, T_flat)

    # Decimate by EXTRA_DECIMATE (10×)
    data = data[..., ::EXTRA_DECIMATE]  # (20, 8, T_100x)
    T_100x = data.shape[-1]
    X = data.reshape(CHANNELS, T_100x).T  # (T_100x, 160)

    # Convert tdisrupt to 100× space relative to flattop start
    if tdis_dec >= 0:
        tdisrupt_100x = max(0, (tdis_dec - flat_start) // EXTRA_DECIMATE)
    else:
        tdisrupt_100x = -1
    return X, tdisrupt_100x


# Quick test: load one disrupt shot
if disrupt_valid:
    _X, _td = load_flattop_100x(DISRUPT_DECIMATED, disrupt_valid[0], is_clear=False)
    print(f"Test shot {disrupt_valid[0]}: X.shape={_X.shape}, tdisrupt_100x={_td}")
if clear_valid:
    _X, _td = load_flattop_100x(CLEAR_DECIMATED, clear_valid[0], is_clear=True)
    print(f"Test shot {clear_valid[0]}: X.shape={_X.shape}, tdisrupt_100x={_td}")

## 4. Streaming Welford PCA — fit on training shots only

Single-pass online Welford algorithm: accumulate mean μ ∈ ℝ¹⁶⁰ and covariance M₂ ∈ ℝ¹⁶⁰×¹⁶⁰ from training flattop data (after extra 10× decimation). Memory stays O(160²). Then eigendecompose and take top 8 PCs.

In [ ]:
BATCH_SIZE = 10_000  # time points per Welford update


def welford_merge(n, mean, M2, batch: np.ndarray):
    """Merge batch (B, 160) into running (n, mean, M2). Memory O(160²)."""
    B = len(batch)
    if B == 0:
        return n, mean, M2
    batch_mean = batch.mean(axis=0)
    batch_M2 = (batch - batch_mean).T @ (batch - batch_mean)
    n_new = n + B
    mean_new = (n * mean + B * batch_mean) / n_new
    delta = mean - batch_mean
    M2_new = M2 + batch_M2 + (n * B / n_new) * np.outer(delta, delta)
    return n_new, mean_new, M2_new


# Accumulate over training shots (disrupt + clear)
n_total = 0
mean = np.zeros(CHANNELS, dtype=np.float64)
M2 = np.zeros((CHANNELS, CHANNELS), dtype=np.float64)

train_tuples = [(DISRUPT_DECIMATED, s, False) for s in disrupt_train] + \
               [(CLEAR_DECIMATED, s, True) for s in clear_train]

for root, shot, is_clear in tqdm(train_tuples, desc="Welford PCA (train only)"):
    try:
        X, _ = load_flattop_100x(root, shot, is_clear)  # (T_100x, 160)
    except (OSError, IOError, KeyError):
        print(f"  SKIP shot {shot} (read error)")
        continue
    for start in range(0, len(X), BATCH_SIZE):
        batch = X[start:start + BATCH_SIZE]
        n_total, mean, M2 = welford_merge(n_total, mean, M2, batch)

print(f"Training time points: {n_total:,}")
assert n_total >= 2, "Need at least 2 training time points"

# Eigendecompose
C = M2 / (n_total - 1)
eigenvalues, eigenvectors = np.linalg.eigh(C)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

mu = mean.astype(np.float32)              # (160,)
W_K = eigenvectors[:, :N_PCA].astype(np.float32)  # (160, 8)
var_explained = eigenvalues / eigenvalues.sum()

print(f"Global PCA: μ ∈ R^{CHANNELS}, W_K ∈ R^{CHANNELS}×{N_PCA}")
print(f"Cumulative variance (top {N_PCA}): {var_explained[:N_PCA].sum():.4f}")
print(f"Per-component: {var_explained[:N_PCA]}")

## 5. Build subsequences and save to single H5

For each shot (all splits): load flattop → decimate 10× → PCA project → tile into subsequences of length `T_SUB=7812` with stride `STRIDE=4810`. Compute per-subsequence target/weight using Twarn logic. Save everything in one H5 file.

**H5 layout:**
```
/{split}/X        (N, 8, 7812) float32
/{split}/target   (N, 7812)    float32
/{split}/weight   (N, 7812)    float32
/{split}/labels   (N,)         int64    — 1=has disruptive label, 0=clear
/{split}/shot_ids (N,)         int64    — source shot ID
/pca/mu           (160,)       float32
/pca/W_K          (160, 8)     float32
/pca/var_explained (8,)        float32
```

In [ ]:
def pca_project(X: np.ndarray) -> np.ndarray:
    """Project (T, 160) → (8, T) float32 using global PCA."""
    Z = (X.astype(np.float32) - mu) @ W_K  # (T, 8)
    return Z.T  # (8, T)


def compute_target_weight(T: int, tdisrupt_100x: int, twarn_100x: int):
    """Compute target and weight for one subsequence.

    Args:
        T: subsequence length
        tdisrupt_100x: disruption onset index within this subsequence; -1 if clear
        twarn_100x: warning window length in 100× samples
    Returns:
        target: (T,) float32 — 1.0 in [disrupt_idx, end), 0.0 elsewhere
        weight: (T,) float32 — 1.0 where loss is active, 0.0 after disruption
    """
    target = np.zeros(T, dtype=np.float32)
    weight = np.ones(T, dtype=np.float32)
    if tdisrupt_100x >= 0 and tdisrupt_100x < T:
        # Positive label from (tdisrupt - twarn) to tdisrupt
        disrupt_start = max(0, tdisrupt_100x - twarn_100x)
        target[disrupt_start:tdisrupt_100x] = 1.0
        # Zero weight after actual disruption (no useful signal)
        weight[tdisrupt_100x:] = 0.0
    elif tdisrupt_100x >= 0 and tdisrupt_100x >= T:
        # Disruption is beyond this window — entire window is pre-disruptive but no positive label
        pass
    # Clear shots: target stays 0, weight stays 1
    return target, weight


def process_shot(root: Path, shot: int, is_clear: bool):
    """Load, PCA-project, tile into subsequences. Returns lists of (X, target, weight, label, shot_id)."""
    try:
        X_raw, tdis_100x = load_flattop_100x(root, shot, is_clear)  # (T_100x, 160)
    except (OSError, IOError, KeyError):
        return []
    X_pca = pca_project(X_raw)  # (8, T_100x)
    T_total = X_pca.shape[1]
    if T_total < T_SUB:
        return []

    twarn_100x = TWARN_RAW // TOTAL_DECIMATE  # 300,000 / 100 = 3000

    results = []
    pos = 0
    while pos + T_SUB <= T_total:
        chunk = X_pca[:, pos:pos + T_SUB]  # (8, T_SUB)
        # tdisrupt relative to this chunk
        if tdis_100x >= 0:
            td_local = tdis_100x - pos
        else:
            td_local = -1
        target, weight = compute_target_weight(T_SUB, td_local, twarn_100x)
        has_disrupt = 1 if (td_local >= 0 and td_local < T_SUB) else 0
        results.append((chunk, target, weight, has_disrupt, shot))
        pos += STRIDE

    # Last window (avoid missing tail)
    last_start = T_total - T_SUB
    if last_start > (pos - STRIDE):
        chunk = X_pca[:, last_start:T_total]
        if tdis_100x >= 0:
            td_local = tdis_100x - last_start
        else:
            td_local = -1
        target, weight = compute_target_weight(T_SUB, td_local, twarn_100x)
        has_disrupt = 1 if (td_local >= 0 and td_local < T_SUB) else 0
        results.append((chunk, target, weight, has_disrupt, shot))

    return results


# Build all subsequences per split
split_map = {
    "train": [(DISRUPT_DECIMATED, s, False) for s in disrupt_train] +
             [(CLEAR_DECIMATED, s, True) for s in clear_train],
    "val":   [(DISRUPT_DECIMATED, s, False) for s in disrupt_val] +
             [(CLEAR_DECIMATED, s, True) for s in clear_val],
    "test":  [(DISRUPT_DECIMATED, s, False) for s in disrupt_test] +
             [(CLEAR_DECIMATED, s, True) for s in clear_test],
}

all_data = {}  # split -> {X, target, weight, labels, shot_ids}
for split_name, shot_tuples in split_map.items():
    X_list, tgt_list, wgt_list, lbl_list, sid_list = [], [], [], [], []
    for root, shot, is_clear in tqdm(shot_tuples, desc=split_name):
        for chunk, target, weight, label, sid in process_shot(root, shot, is_clear):
            X_list.append(chunk)
            tgt_list.append(target)
            wgt_list.append(weight)
            lbl_list.append(label)
            sid_list.append(sid)
    if X_list:
        all_data[split_name] = {
            "X": np.stack(X_list),                          # (N, 8, T_SUB)
            "target": np.stack(tgt_list),                   # (N, T_SUB)
            "weight": np.stack(wgt_list),                   # (N, T_SUB)
            "labels": np.array(lbl_list, dtype=np.int64),   # (N,)
            "shot_ids": np.array(sid_list, dtype=np.int64), # (N,)
        }
        n_pos = sum(lbl_list)
        print(f"  [{split_name}] {len(X_list)} subsequences ({n_pos} disruptive, {len(X_list) - n_pos} clear)")
    else:
        print(f"  [{split_name}] 0 subsequences")

## 6. Write single H5 file

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Compute class weights from training split
if "train" in all_data:
    train_targets = all_data["train"]["target"]
    train_weights_mask = all_data["train"]["weight"]
    total_pos = float((train_targets * train_weights_mask).sum())
    total_neg = float(((1 - train_targets) * train_weights_mask).sum())
    total = total_pos + total_neg
    pos_weight = float(0.5 * total / total_pos) if total_pos > 0 else 1.0
    neg_weight = float(0.5 * total / total_neg) if total_neg > 0 else 1.0
    print(f"Class weights: pos_weight={pos_weight:.4f}, neg_weight={neg_weight:.4f}")
else:
    pos_weight = neg_weight = 1.0

with h5py.File(OUT_H5, "w") as f:
    # Per-split datasets
    for split_name, data in all_data.items():
        g = f.create_group(split_name)
        g.create_dataset("X", data=data["X"], dtype=np.float32)
        g.create_dataset("target", data=data["target"], dtype=np.float32)
        g.create_dataset("weight", data=data["weight"], dtype=np.float32)
        g.create_dataset("labels", data=data["labels"], dtype=np.int64)
        g.create_dataset("shot_ids", data=data["shot_ids"], dtype=np.int64)

    # PCA parameters (for inverse transform / inspection)
    g_pca = f.create_group("pca")
    g_pca.create_dataset("mu", data=mu)
    g_pca.create_dataset("W_K", data=W_K)
    g_pca.create_dataset("var_explained", data=var_explained[:N_PCA].astype(np.float32))

    # Metadata as attributes on root
    f.attrs["nsub_raw"] = NSUB_RAW
    f.attrs["stride_raw"] = STRIDE_RAW
    f.attrs["data_step"] = DATA_STEP
    f.attrs["extra_decimate"] = EXTRA_DECIMATE
    f.attrs["total_decimate"] = TOTAL_DECIMATE
    f.attrs["t_sub"] = T_SUB
    f.attrs["stride"] = STRIDE
    f.attrs["n_pca"] = N_PCA
    f.attrs["twarn_ms"] = TWARN_MS
    f.attrs["pos_weight"] = pos_weight
    f.attrs["neg_weight"] = neg_weight
    f.attrs["random_seed"] = RANDOM_SEED
    for split_name in all_data:
        f.attrs[f"n_{split_name}"] = len(all_data[split_name]["labels"])

print(f"\nSaved to {OUT_H5}")
# File size
import os
size_mb = os.path.getsize(OUT_H5) / (1024 ** 2)
print(f"File size: {size_mb:.1f} MB")

## 7. Verify: shapes, label distribution, quick sanity plot

In [ ]:
# Read back and verify
with h5py.File(OUT_H5, "r") as f:
    print("=== H5 structure ===")
    for key in f.keys():
        if isinstance(f[key], h5py.Group):
            print(f"  /{key}/")
            for dset_name in f[key]:
                print(f"    {dset_name}: {f[key][dset_name].shape} {f[key][dset_name].dtype}")

    print("\n=== Metadata ===")
    for attr in f.attrs:
        print(f"  {attr}: {f.attrs[attr]}")

    print("\n=== Split stats ===")
    for split in ("train", "val", "test"):
        if split not in f:
            continue
        g = f[split]
        X = g["X"]
        labels = np.asarray(g["labels"])
        n_total = len(labels)
        n_pos = int(labels.sum())
        print(f"  {split}: {n_total} subseqs, X.shape={X.shape}, "
              f"disruptive={n_pos}, clear={n_total - n_pos}")

    # Quick sanity: first train subsequence
    if "train" in f:
        X0 = np.asarray(f["train/X"][0])
        t0 = np.asarray(f["train/target"][0])
        w0 = np.asarray(f["train/weight"][0])
        print(f"\n=== First train subseq ===")
        print(f"  X: shape={X0.shape}, min={X0.min():.3f}, max={X0.max():.3f}")
        print(f"  target: {t0.sum():.0f} positive samples / {len(t0)}")
        print(f"  weight: nonzero={int((w0 > 0).sum())} / {len(w0)}")

In [ ]:
import matplotlib.pyplot as plt

# Plot a disruptive and a clear subsequence from train split
with h5py.File(OUT_H5, "r") as f:
    if "train" not in f:
        raise RuntimeError("No train split")
    labels = np.asarray(f["train/labels"])
    pos_idxs = np.where(labels == 1)[0]
    neg_idxs = np.where(labels == 0)[0]

    fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex=True)

    for col, (name, idx) in enumerate([
        ("Disruptive", pos_idxs[0] if len(pos_idxs) else 0),
        ("Clear", neg_idxs[0] if len(neg_idxs) else 0),
    ]):
        X = np.asarray(f["train/X"][idx])        # (8, T)
        tgt = np.asarray(f["train/target"][idx])  # (T,)
        shot = int(f["train/shot_ids"][idx])

        ax_top = axes[0, col]
        for pc in range(min(4, N_PCA)):
            ax_top.plot(X[pc], alpha=0.7, label=f"PC{pc}")
        ax_top.set_title(f"{name} — shot {shot}")
        ax_top.legend(fontsize=7, ncol=4)
        ax_top.set_ylabel("PCA value")

        ax_bot = axes[1, col]
        ax_bot.plot(tgt, color="red", linewidth=1.5)
        ax_bot.set_ylabel("Target")
        ax_bot.set_xlabel("Time (10 kHz samples)")
        ax_bot.set_ylim(-0.1, 1.1)

    plt.tight_layout()
    plt.show()

## 8. Dataloader usage example

Minimal PyTorch Dataset that loads the entire H5 into memory.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class EceiPCA8Dataset(Dataset):
    """Load entire PCA-8 100× flattop H5 into memory."""

    def __init__(self, h5_path: str, split: str = "train"):
        with h5py.File(h5_path, "r") as f:
            g = f[split]
            self.X = torch.from_numpy(np.asarray(g["X"]))          # (N, 8, T)
            self.target = torch.from_numpy(np.asarray(g["target"]))  # (N, T)
            self.weight = torch.from_numpy(np.asarray(g["weight"]))  # (N, T)
            self.labels = np.asarray(g["labels"])                    # (N,)
            self.pos_weight = float(f.attrs.get("pos_weight", 1.0))
            self.neg_weight = float(f.attrs.get("neg_weight", 1.0))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.target[idx], self.weight[idx]


# Test it
ds = EceiPCA8Dataset(str(OUT_H5), "train")
print(f"Train dataset: {len(ds)} samples")
x, t, w = ds[0]
print(f"  x: {x.shape}, target: {t.shape}, weight: {w.shape}")
print(f"  pos_weight={ds.pos_weight:.4f}, neg_weight={ds.neg_weight:.4f}")

dl = DataLoader(ds, batch_size=32, shuffle=True)
batch = next(iter(dl))
print(f"  Batch: X={batch[0].shape}, target={batch[1].shape}, weight={batch[2].shape}")